# A4. Sampler steps

How few sampler steps are enough?

See `docs/plans/experiments-plan.md`, entry A4. Results: `artifacts/results/dla_steps/`.

## Run

From the repo root. `uv run erosion-sim list` shows what is already trained and evaluated.

**1. Data.** Skip any that already exist.

Local:

```sh
uv run erosion-sim dla-generate dla_master
```

Cluster:

```sh
sbatch slurm/dla_generate.sbatch dla_master
```

**2. Train**, in this order. Each resumes if stopped; a finished model is skipped.

Local:

```sh
uv run erosion-sim train dla_base
uv run erosion-sim train dla_flow
```

Cluster:

```sh
sbatch slurm/train.sbatch dla_base
sbatch slurm/train.sbatch dla_flow
```

**3. Evaluate**, once training has finished. It refuses to replace existing results.

Local:

```sh
uv run erosion-sim evaluate dla_steps dla_base
uv run erosion-sim evaluate dla_steps dla_flow
```

Cluster:

```sh
sbatch slurm/evaluate.sbatch dla_steps dla_base
sbatch slurm/evaluate.sbatch dla_steps dla_flow
```

Then run the cells below.

How to read the tables and plots: [`docs/guides/dla.md`](../../docs/guides/dla.md).

In [ ]:
import json

from erosion_sim.dla import plots
from erosion_sim.evaluation.results import load
from erosion_sim.paths import model_dir
from erosion_sim.training.train import plot_history

EVALUATION = "dla_steps"  # configs/evaluations/dla_steps.yaml
PREDICTORS = ["dla_base", "dla_flow"]

results = load(EVALUATION, *PREDICTORS)
{p: {k: m[k] for k in ("time", "code", "slurm_job")} for p, m in results.manifest["predictors"].items()}

## Distance to the simulator against network calls

Each point is a metric's 1-Wasserstein distance to the simulator, averaged over the test conditions. Lower is better.

In [ ]:
plots.plot_steps(results)

In [ ]:
plots.table(results, ["particles", "connected_fraction", "fractal_dimension"])

## Conclusion

_Written after reading the results above._

A5 and A6 use `dla_test` and `dla_agreement`, at 32 steps. For another step count, add evaluation files with that `steps` and use them there.